# Setup

Change `/workspaces/taegis-sdk-python-workspace/taegis-magic` to wherever your taegis-magic dir is that is on the correct branch.<br><br>
After changing the directory make sure to reload the kernel!

In [1]:
!pip install -e /Users/chris.langreo/dev/taegis/taegis-magic-dir/taegis-magic

Obtaining file:///Users/chris.langreo/dev/taegis/taegis-magic-dir/taegis-magic
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for taegis-magic (pyproject.toml) ... done
  Created wheel for taegis-magic: filename=taegis_magic-2026.3.25-py3-none-any.whl size=10234 sha256=55ac65aa32261a8308ac566037038f21c364237c01f06c8b6db91d15e9f14860
  Stored in directory: /private/var/folders/s2/lksr8fqd63x_h9t2bzmj5yw80000gp/T/pip-ephem-wheel-cache-ih6q3txr/wheels/d4/57/82/dbe191281b1fbf322b3f0bd5f6fa278bba8a728b9c6cf9f4ff
Successfully built taegis-magic
  Attempting uninstall: taegis-magic
    Found existing installation: taegis-magic 2026.3.25
    Uninstalling taegis-magic-2026.3.25:
      Successfully uninstalled taegis-magic-2026.3.25

[notice] A new release of pip is available

Installing itables for the purposes of displaying DataFrames in a nicer format

In [2]:
!pip install itables


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from taegis_magic.pandas.process import process_correlate_netflow
import pandas as pd
from itables import show


In [4]:
%load_ext taegis_magic

In [5]:
%taegis auth login --use-universal-authentication

Login for region: universal
Copy URL into a browser: https://api.taegis.secureworks.com/auth/device/code/activate?user_code=2NJR-QL5X


**Taegis Results**

|Region          |Tenant             |Service          |Total Results                       |
|----------------|-------------------|-----------------|------------------------------------|
|None|None|authentication|1|

The query below can be accessed via: https://ctpx.secureworks.com/share/c88dab66-7876-44c8-998f-a7ca97851e8f

The query specifies two process_correlation_ids where each one correlates to a different netflow event. One netflow event has a `processcorrelationid.pid` with a format of `pid:timewindow` with an empty `processcorrelationid.timewindow`, and the other a `processcorrelationid.pid` with a format of `pid` with a non empty `processcorrelationid.timewindow` value.

The query is just to get some valid process data to put into the new function, `process_correlate_netflow`

In [14]:
%%taegis events search --assign process_events
from process
WHERE process_correlation_id in ('302c8cb3-7ae4-5264-907e-6716ce554652:25614222880669775:1774877809989093', '302c8cb3-7ae4-5264-907e-6716ce554652:25614222880671395:1774896303136505')
EARLIEST = -30d

Input field `saveToCache` is deprecated: 'Moving to backendStrategy as an exclusion option'
Input field `searchTarget` is deprecated: 'Moved to backendStrategy; ignored'
Output field `backend` is deprecated: 'Moved to backendSource', removing from default output...
Output field `name` is deprecated: 'name is deprecated. Use `userID` instead.', removing from default output...
Output field `backend` is deprecated: 'Moved to backendSource', removing from default output...


**Taegis Search Results**

ID: *qlquery://1775136759:ed2c5a3e-5395-4bcf-9a0a-9e0e317663db*



|Region          |Tenant             |Service          |Status          |Num. Total                          |Num. Returned                          |Link                   |
|----------------|-------------------|-----------------|----------------|------------------------------------|---------------------------------------|:----------------------|
|charlie|None|events|SUCCEEDED|N/A|2|https://ctpx.secureworks.com/share/f60d7299-393e-4187-90ab-26bd3c321e22|


# Beginning of Testing

Edit the `process_events` variable returned by the above query to only contain the process_correlation_id column for the sake of simplicity/readability. 

In [7]:
process_events = process_events[["process_correlation_id"]]

In [15]:
print(process_events)

                                         commandline commandline_decoded  \
0       C:\Windows\System32\svchost.exe -k utcsvc -p                       
1  "\Device\HarddiskVolume3\Program Files\Microso...                       

  computer_name                                      enrichSummary  \
0                     C:\Windows\System32\svchost.exe -k utcsvc -p   
1                "\Device\HarddiskVolume3\Program Files\Microso...   

   enrichedHostnames event_time_fidelity   event_time_usec  \
0               True               MICRO  1774877818726780   
1               True               MICRO  1774896303139372   

   existed_before_agent gid      hidden  ... program_hash.md5  \
0                 False      NOT_HIDDEN  ...                    
1                 False      NOT_HIDDEN  ...                    

  program_hash.sha1                                program_hash.sha256  \
0                    0AD27DC6B692903C4E129B1AD75EE8188DA4B9CE34C309...   
1                    48EAC133

Setting log level to DEBUG to show more detail if desired

In [9]:
# import logging
# logging.basicConfig(level=logging.DEBUG)
# logging.getLogger("taegis_magic").setLevel(logging.DEBUG)

In [10]:
region="charlie"
tenant_id="11063"

## Testing Valid Data

Now calling new function, `process_correlate_netflow` with valid data

In [16]:
ret_valid_data = process_events.pipe(process_correlate_netflow, region=region, tenant_id=tenant_id, earliest='30d')

Input field `saveToCache` is deprecated: 'Moving to backendStrategy as an exclusion option'
Input field `searchTarget` is deprecated: 'Moved to backendStrategy; ignored'
Output field `backend` is deprecated: 'Moved to backendSource', removing from default output...
Output field `name` is deprecated: 'name is deprecated. Use `userID` instead.', removing from default output...
Output field `backend` is deprecated: 'Moved to backendSource', removing from default output...


In [12]:
print(len(ret_valid_data))

100


Note, that while there were originally only two rows in the input DataFrame that more rows may be returned due to the 1:many relationship between process events and netflow events. 

In [13]:
show(ret_valid_data)

Loading ITables v2.7.1 from the internet... (need help?)


## Testing putting DataFrame returned by `process_correlate_netflow` as input to the function
In this case, it should just return the DataFrame that was put into the function

In [ ]:
process_correlate_netflow(ret_valid_data, region="charlie", tenant_id=tenant_id, earliest="10d")

## Testing providing a different value for `process_column`

First, will create a copy of the `process_events` from earlier and change the name

In [ ]:
new_process_column = "my_pid"
process_events_diff_col = process_events.copy()
process_events_diff_col.rename(columns={"process_correlation_id":new_process_column},inplace=True)

In [ ]:
print(process_events_diff_col)

In [ ]:
ret_with_diff_process_column = process_events_diff_col.pipe(process_correlate_netflow, region=region, tenant_id=tenant_id, process_column=new_process_column, earliest="10d")

In [ ]:
show(ret_with_diff_process_column)

## Testing where `process_column` is not in DataFrame

In [ ]:
ret_valid_data = process_events.pipe(process_correlate_netflow, region=region, tenant_id=tenant_id, process_column=new_process_column)